# DOH Vision · WHAM 3D 회전 (Tier-B)

**목표:** 단일 골프 스윙 영상 → WHAM(SMPL 3D 복원) → 흉곽/골반 회전을 **실제 각도**로.

브라우저(analyzer2 / MediaPipe)에서 정면 백스윙탑 흉곽이 -19°로 붕괴한 이유는 단일프레임 깊이(z)가 약해서였다.
WHAM은 인체 프라이어 + 시간축으로 3D를 복원하므로, **같은 azimuth 수식**을 정확한 3D에 먹이면 실제(~80~90°)에 근접해야 한다.
이 노트북의 목적은 그 가설을 **형의 실제 스윙 영상으로 검증**하는 것이다 (Phase 1).

> ⚠️ **런타임 → GPU** 로 바꾸세요 (런타임 유형 변경 → T4 GPU).
> WHAM 공식 Colab: https://colab.research.google.com/drive/1ysUtGSwidTQIdBQRhq0hj63KbseFujkn
> SMPL 모델은 라이선스 등록 필요: https://smpl.is.tue.mpg.de / https://smplify.is.tue.mpg.de


## 1. WHAM 설치
공식 절차. `fetch_demo_data.sh`는 SMPL 등록 자격증명이 필요하다(스크립트 안내 따름).
설치가 까다로우면 위 **공식 Colab**에서 pkl만 뽑아 4번으로 건너뛰어도 된다.


In [ ]:
# WHAM clone + 설치 (공식 docs/INSTALL.md 기준)
!git clone https://github.com/yohanshin/WHAM.git
%cd WHAM
!git submodule update --init --recursive
# 의존성 (버전 충돌 시 공식 INSTALL.md/environment 확인)
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt || echo 'requirements 일부 실패 시 개별 설치'
# 체크포인트 + SMPL 바디모델 다운로드 (SMPL 계정 필요)
!bash fetch_demo_data.sh


## 2. 스윙 영상 업로드
정면(FO) 또는 측면(DTL) 스윙 mp4. 짧게(스윙 구간만) 자른 게 빠르다.


In [ ]:
from google.colab import files
up = files.upload()                 # 스윙 mp4 선택
VIDEO = list(up.keys())[0]
print('업로드:', VIDEO)


## 3. WHAM 추론
카메라 좌표만 필요하면 `--estimate_local_only`(SLAM 생략, 빠름). 
월드좌표(중력정렬) 원하면 옵션 제거. 결과 pkl은 `output/demo/<name>/` 아래 저장됨.


In [ ]:
!python demo.py --video "{VIDEO}" --save_pkl --estimate_local_only --visualize

# 산출 pkl 찾기
import glob, os
pkls = sorted(glob.glob('output/demo/**/*.pkl', recursive=True), key=os.path.getmtime)
assert pkls, 'pkl을 못 찾음 — demo 로그 확인'
PKL = pkls[-1]; print('결과 pkl:', PKL)


## 4. DOH 회전 추출
먼저 `--check`로 pkl의 키/shape를 눈으로 확인(‘joints’가 (T,J,3)인지). 
SMPL-24 순서가 아니면 `wham_golf_rotation.py`의 `SMPL` 인덱스만 맞추면 된다.


In [ ]:
# DOH 분석 스크립트 가져오기
!wget -q https://raw.githubusercontent.com/tinyalex3628-dotcom/doh-golf-survey/claude/doh-vision-handoff-review-j1rim1/pose3d_poc/wham_golf_rotation.py -O wham_golf_rotation.py
!python wham_golf_rotation.py "{PKL}" --check


In [ ]:
# P구간 프레임을 대략 넣고(영상 보고), 회전량 확인 + 그래프
# 예: 어드레스 P1=?, 백스윙탑 P4=?, 임팩트 P7=?
!python wham_golf_rotation.py "{PKL}" --p1 0 --p4 0 --p7 0 --png rot.png
from IPython.display import Image; import os
Image('rot.png') if os.path.exists('rot.png') else print('P4 프레임을 넣으면 그래프가 나옵니다')


## 5. 판정
- **백스윙탑 흉곽 회전이 60~90°** 근처로 나오면 → WHAM 접근 **성공**, Tier-B 파이프라인으로 승격.
- 여전히 20~30° 이하로 작으면 → WHAM도 이 영상/앵글에선 부족 → HybrIK/4D-Humans 비교 또는 2카메라 검토.

결과 숫자를 기록해서 브라우저(analyzer2)·실제 스윙과 3자 대조.
